# Lesson 10.2: How Do You Route Queries Between Different Retrieval Strategies?

**Companion notebook for Lesson 10.2 — Module 10: Agentic RAG**

---

| Section | What you will build |
|---|---|
| 1. The Multi-Backend Problem | Prove wrong routing causes hallucination |
| 2. Mock Backends | Four realistic fake backends (vector, SQL, web, API) |
| 3. Router 1 — Keyword | Rules-based pattern matching; test suite with accuracy |
| 4. Router 2 — LLM | Intent classification via prompt; mock + real Claude |
| 5. Router 3 — Embedding | Exemplar-based cosine similarity; confidence trap demo |
| 6. Router Comparison | Accuracy / speed / cost chart across all three |
| 7. Fan-out Pattern | Query all backends in parallel; merge + rerank |
| 8. Latency Math | Serial router vs. parallel fan-out; timing chart |
| 9. Hybrid Router | Keyword/embedding for obvious queries; fan-out for ambiguous |
| 10. Tool-call Routing | Claude tool use: pick backend AND generate parameters |
| 11. Decision Framework | When to use each router; cost-of-wrong-route guide |

**Required:** `sentence-transformers`, `numpy`, `matplotlib`  
**Optional (Sections 4, 10):** `anthropic`  

> LLM-dependent sections run in **mock mode** by default (`USE_REAL_LLM = False`).
> Set `ANTHROPIC_API_KEY` and `USE_REAL_LLM = True` to run with live Claude calls.


In [ ]:
# Uncomment to install
# !pip install sentence-transformers numpy matplotlib
# !pip install anthropic   # optional

%matplotlib inline
import os, json, time, re
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from concurrent.futures import ThreadPoolExecutor, as_completed

os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ['OMP_NUM_THREADS']        = '1'

plt.rcParams['figure.figsize'] = (14, 5)
plt.rcParams['font.size']      = 11
plt.rcParams['axes.grid']      = True
plt.rcParams['grid.alpha']     = 0.3

USE_REAL_LLM = False  # set True + ANTHROPIC_API_KEY to enable live calls

ANTHROPIC_AVAILABLE = False
try:
    import anthropic as _anthropic_module
    ANTHROPIC_AVAILABLE = bool(os.environ.get('ANTHROPIC_API_KEY'))
except ImportError:
    pass

if USE_REAL_LLM and not ANTHROPIC_AVAILABLE:
    print('WARNING: USE_REAL_LLM=True but anthropic not available. Falling back to mock.')
    USE_REAL_LLM = False

print(f'LLM mode: {"REAL (Claude API)" if USE_REAL_LLM else "MOCK (deterministic)"}')
print('Imports ready.')


In [ ]:
# ── Mock backend implementations ─────────────────────────────────────────────
# Each backend returns (result_text, latency_ms).
# In production: swap these for real Pinecone, psycopg2, requests, etc.

VECTOR_STORE = [
    {'id':'doc1', 'text':'Our refund policy: refunds within 14 days of purchase. Digital goods non-refundable.'},
    {'id':'doc2', 'text':'Pro plan includes: unlimited storage, priority support, API access.'},
    {'id':'doc3', 'text':'Enterprise SLA: 99.9% uptime, 4-hour response time, dedicated account manager.'},
    {'id':'doc4', 'text':'We had record revenue in Q3, driven by strong enterprise sales growth.'},
    {'id':'doc5', 'text':'Shipping: free standard shipping over $50. Delivery in 3-5 business days.'},
]

SQL_DATABASE = {
    'monthly_orders': {'Jan':1243,'Feb':1389,'Mar':1521,'Apr':1670,'May':1830,'Jun':2001},
    'revenue_q3':     {'Jul':842000,'Aug':917000,'Sep':1034000,'total':2793000},
    'top_customers':  [('Acme Corp',45800),('TechStart',38200),('GlobalMfg',31500)],
    'avg_order_value':{'2024':127.50,'2025':143.20},
    'churn_rate':     {'Q1':0.032,'Q2':0.028,'Q3':0.031,'Q4':0.025},
}

WEB_ARTICLES = [
    {'date':'2026-06-20','title':'TechCo announces new AI-powered dashboard',
     'snippet':'CEO Jane Smith unveiled the feature at the annual summit...'},
    {'date':'2026-06-18','title':'Competitor launches competing product at lower price',
     'snippet':'Rival firm cut prices by 15%, targeting mid-market segment...'},
    {'date':'2026-06-15','title':'Industry report: SaaS churn trends in 2026',
     'snippet':'Average churn rates have fallen for the third consecutive quarter...'},
]

LIVE_API = {
    'TSLA':  {'price':248.93,'change':+1.2,'volume':32_400_000},
    'GOOGL': {'price':178.42,'change':-0.5,'volume':18_700_000},
    'weather_NYC': {'temp_f':71,'condition':'Partly cloudy','humidity':58},
    'weather_LON': {'temp_f':59,'condition':'Overcast','humidity':75},
}


def backend_vector(query: str) -> tuple:
    t0 = time.perf_counter()
    words  = set(query.lower().split())
    scored = sorted(VECTOR_STORE,
                    key=lambda d: len(words & set(d['text'].lower().split())),
                    reverse=True)
    result = scored[0]['text'] if scored else 'No relevant documents found.'
    return result, int((time.perf_counter() - t0) * 1000 + 50)  # +50ms simulated


def backend_sql(query: str) -> tuple:
    t0 = time.perf_counter()
    q  = query.lower()
    if 'revenue' in q and 'q3' in q:
        d = SQL_DATABASE['revenue_q3']
        result = f'Q3 Revenue: Jul ${d["Jul"]:,} | Aug ${d["Aug"]:,} | Sep ${d["Sep"]:,} | Total ${d["total"]:,}'
    elif 'order' in q:
        d = SQL_DATABASE['monthly_orders']
        result = 'Monthly orders: ' + ', '.join(f'{m}: {n}' for m, n in d.items())
    elif 'customer' in q:
        rows = SQL_DATABASE['top_customers']
        result = 'Top customers: ' + ', '.join(f'{n} (${v:,})' for n, v in rows)
    elif 'churn' in q:
        d = SQL_DATABASE['churn_rate']
        result = 'Churn rates: ' + ', '.join(f'{q}: {r:.1%}' for q, r in d.items())
    else:
        result = f'SQL query executed. No matching table for: "{query}"'
    return result, int((time.perf_counter() - t0) * 1000 + 80)


def backend_web(query: str) -> tuple:
    t0 = time.perf_counter()
    words   = set(query.lower().split())
    scored  = sorted(WEB_ARTICLES,
                     key=lambda a: len(words & set((a['title']+' '+a['snippet']).lower().split())),
                     reverse=True)
    art     = scored[0]
    result  = f'[{art["date"]}] {art["title"]}: {art["snippet"]}'
    return result, int((time.perf_counter() - t0) * 1000 + 300)  # web is slower


def backend_api(query: str) -> tuple:
    t0     = time.perf_counter()
    q      = query.lower()
    result = None
    for ticker in ['TSLA', 'GOOGL']:
        if ticker.lower() in q:
            d      = LIVE_API[ticker]
            result = f'{ticker}: ${d["price"]} (change: {d["change"]:+.1f}%, vol: {d["volume"]:,})'
    for city_key, city_name in [('weather_NYC','New York'), ('weather_LON','London')]:
        if city_name.lower() in q or city_key in q:
            d      = LIVE_API[city_key]
            result = f'{city_name} weather: {d["temp_f"]}F, {d["condition"]}, humidity {d["humidity"]}%'
    if not result:
        result = 'API: no matching real-time data found.'
    return result, int((time.perf_counter() - t0) * 1000 + 120)


BACKENDS = {
    'vector_store': backend_vector,
    'sql':          backend_sql,
    'web_search':   backend_web,
    'api':          backend_api,
}

print('Mock backends ready:')
for name in BACKENDS:
    print(f'  {name}')


---
## 1. The Multi-Backend Problem — Wrong Routing Causes Hallucination

Each backend speaks a different language:

| Backend | Strength | Blind spot |
|---|---|---|
| Vector store | Semantic similarity, "what is similar to X?" | Cannot count, aggregate, or give exact numbers |
| SQL database | Exact counts, aggregations, structured joins | Cannot reason about unstructured text |
| Web search | Current events, yesterday's news | Cannot answer from your private documents |
| Live API | Real-time data (prices, weather) | No history, no documents |

If a user asks **"Show me last week's revenue"** and your code sends it to the vector store,
you'll retrieve a marketing PDF that says *"We had record revenue!"* —
and the LLM will hallucinate a specific number from that vague phrase.

> **Routing is deciding: for this query, which backend(s) should we call?**


In [ ]:
# Demonstrate exactly why wrong routing is dangerous

SQL_QUERIES = [
    'Show me last week revenue',
    'How many orders did we get this month?',
    'What is the average order value in 2025?',
    'Which customers spent the most?',
]

print('=== Wrong routing: SQL queries sent to vector store ===\n')
print('What the LLM "sees" (and may hallucinate from):\n')

for q in SQL_QUERIES:
    wrong_result, _   = backend_vector(q)  # wrong backend!
    correct_result, _ = backend_sql(q)     # right backend
    print(f'Query  : "{q}"')
    print(f'WRONG  (vector): {wrong_result[:120]}')
    print(f'CORRECT (sql)  : {correct_result[:120]}')
    print()

print('The vector store returns a marketing sentence about "record revenue".')
print('The LLM gets no actual number to return, so it fabricates one.')
print('This is a confident hallucination — no error raised, no warning logged.')
print()
print('Routing is the gate that prevents this. Get the gate wrong once and')
print('the user gets a convincingly wrong answer.')


---
## 3. Router 1 — Keyword Router (Rules-Based)

The simplest possible approach: pattern-match on the query.

**Pros:** zero latency, zero cost, fully deterministic.  
**Cons:** brittle — every new phrasing needs a new rule.

> **The hidden cost:** keyword routers look free, but after 6 months you have
> a 200-line `if/elif` ladder that nobody dares touch. The "deterministic" advantage
> fades once the rules contradict each other.

**Use it when:** your queries are narrow and predictable (e.g., a finance dashboard
with a known query vocabulary).


In [ ]:
SQL_KEYWORDS = [
    'select', 'count', 'how many', 'average', 'avg', 'total', 'sum',
    'revenue', 'orders', 'customers', 'churn', 'last week', 'last month',
    'last quarter', 'this month', 'per customer', 'conversion rate',
]
WEB_KEYWORDS  = ['latest', 'today', 'yesterday', 'recent news', 'announced', 'breaking']
API_KEYWORDS  = ['price', 'stock', 'weather', 'current temperature', 'live', 'right now']


def keyword_router(query: str) -> str:
    q = query.lower()
    if any(kw in q for kw in SQL_KEYWORDS):
        return 'sql'
    if any(kw in q for kw in WEB_KEYWORDS):
        return 'web_search'
    if any(kw in q for kw in API_KEYWORDS):
        return 'api'
    return 'vector_store'


# ── Test suite ────────────────────────────────────────────────────────────────
TEST_CASES = [
    # (query, expected_backend, difficulty)
    ('How does our refund policy work?',                     'vector_store', 'easy'),
    ('What features does the Pro plan include?',             'vector_store', 'easy'),
    ('How many orders did we get this month?',               'sql',          'easy'),
    ('What was the total revenue in Q3?',                    'sql',          'easy'),
    ('What did our CEO announce yesterday?',                 'web_search',   'easy'),
    ('What is the current TSLA stock price?',                'api',          'easy'),
    ('What is the weather in New York right now?',           'api',          'easy'),
    # Hard cases: natural phrasing without obvious keywords
    ('Show me our numbers for last quarter',                 'sql',          'hard'),
    ('Did our competitor release anything this week?',       'web_search',   'hard'),
    ('How are we doing financially?',                        'sql',          'hard'),
    ('Give me the breakdown of customer spend',              'sql',          'hard'),
    ('Are there any issues with our enterprise SLA?',        'vector_store', 'hard'),
]


print(f'{"Query":<55} {"Expected":<14} {"Got":<14} {"OK?"}')
print('-' * 95)

correct_easy = correct_hard = total_easy = total_hard = 0
kw_results = []
for q, expected, difficulty in TEST_CASES:
    got    = keyword_router(q)
    ok     = got == expected
    kw_results.append({'query': q, 'expected': expected, 'got': got,
                       'correct': ok, 'difficulty': difficulty})
    if difficulty == 'easy':
        total_easy += 1
        correct_easy += ok
    else:
        total_hard += 1
        correct_hard += ok
    mark = 'OK' if ok else '!'
    print(f'{q[:53]:<55} {expected:<14} {got:<14} {mark}')

print()
print(f'Easy queries: {correct_easy}/{total_easy} = {correct_easy/total_easy:.0%}')
print(f'Hard queries: {correct_hard}/{total_hard} = {correct_hard/total_hard:.0%}')
print(f'Overall     : {correct_easy+correct_hard}/{len(TEST_CASES)} = {(correct_easy+correct_hard)/len(TEST_CASES):.0%}')
print()
print('The keyword router handles predictable phrasings but misses natural-language variants.')
print('"Show me our numbers for last quarter" has no SQL keyword — goes to vector_store by default.')


---
## 4. Router 2 — LLM Router (Intent Classification)

Hand the query to an LLM and ask it to pick a backend.
Same idea as Lesson 10.1's planner — LLM as a decision-maker.

**Pros:** handles fuzzy natural-language queries; easy to extend (edit the prompt).  
**Cons:** adds ~200–800 ms before retrieval even starts; costs money; harder to debug.

> **Production tip:** always log the query, the chosen backend, and the model's reasoning.
> When a user complains, you need that log to know whether routing or retrieval was wrong.
> Without logs, debugging an LLM router feels like reading tea leaves.

**Use it when:** queries are diverse and natural-language; you can afford the extra latency.


In [ ]:
LLM_ROUTER_PROMPT = """\
You are a query router for a company knowledge system. Given the user query,
pick the SINGLE best backend to answer it.

Backends:
- "sql": questions about structured data (orders, revenue, customer counts,
  churn rates, averages, totals — anything that requires a precise number)
- "vector_store": questions about product documentation, policies, how-tos,
  feature descriptions (anything in our internal knowledge base)
- "web_search": questions about current events, news published today or recently,
  competitor activity, press releases (requires fresh information from the web)
- "api": questions needing real-time live data (current stock prices, live weather)

Query: {query}

Respond with ONLY the backend name. No explanation."""

# ── Mock LLM router (deterministic, no API key) ───────────────────────────────
MOCK_LLM_ROUTES = {
    'How does our refund policy work?':               'vector_store',
    'What features does the Pro plan include?':        'vector_store',
    'How many orders did we get this month?':          'sql',
    'What was the total revenue in Q3?':               'sql',
    'What did our CEO announce yesterday?':            'web_search',
    'What is the current TSLA stock price?':           'api',
    'What is the weather in New York right now?':      'api',
    'Show me our numbers for last quarter':            'sql',       # hard — LLM gets it right
    'Did our competitor release anything this week?':  'web_search',# hard — LLM gets it right
    'How are we doing financially?':                   'sql',       # hard — LLM gets it right
    'Give me the breakdown of customer spend':         'sql',       # hard — LLM gets it right
    'Are there any issues with our enterprise SLA?':   'vector_store', # hard — LLM gets it right
}


def llm_router(query: str) -> tuple:
    """Returns (backend, latency_ms)."""
    t0 = time.perf_counter()

    if USE_REAL_LLM:
        client = _anthropic_module.Anthropic()
        resp   = client.messages.create(
            model='claude-haiku-4-5-20251001',
            max_tokens=10,
            messages=[{'role': 'user',
                       'content': LLM_ROUTER_PROMPT.format(query=query)}],
        )
        backend = resp.content[0].text.strip().lower()
    else:
        time.sleep(0.02)   # simulate ~20 ms mock latency
        backend = MOCK_LLM_ROUTES.get(query, 'vector_store')

    latency_ms = int((time.perf_counter() - t0) * 1000)
    return backend, latency_ms


print('=== LLM Router test suite ===\n')
print(f'{"Query":<55} {"Expected":<14} {"Got":<14} {"OK?"}')
print('-' * 95)

llm_correct = 0
llm_results = []
for q, expected, difficulty in TEST_CASES:
    got, lat = llm_router(q)
    ok  = got == expected
    if ok:
        llm_correct += 1
    llm_results.append({'query': q, 'expected': expected, 'got': got,
                        'correct': ok, 'difficulty': difficulty, 'latency_ms': lat})
    mark = 'OK' if ok else '!'
    print(f'{q[:53]:<55} {expected:<14} {got:<14} {mark}')

print()
print(f'LLM router accuracy: {llm_correct}/{len(TEST_CASES)} = {llm_correct/len(TEST_CASES):.0%}')
kw_correct = sum(1 for r in kw_results if r['correct'])
print(f'Keyword accuracy   : {kw_correct}/{len(TEST_CASES)} = {kw_correct/len(TEST_CASES):.0%}')
print()
print('The LLM router handles "Show me our numbers for last quarter" correctly')
print('because it understands intent, not just keywords.')


---
## 5. Router 3 — Embedding Router (Exemplar-Based)

Cheaper than an LLM call, smarter than keywords:

1. Write **5–10 example queries** for each backend
2. **Embed them** once at startup
3. At runtime: embed the user query, find the **closest exemplar**, route to that backend

```
"How many customers do we have?"  →  embed  →  vector
                                                ↓
closest match: "How many orders this month?" (sql exemplar) → route to sql
```

**Pros:** ~50ms (embedding only); cheap; tune by adding exemplars.  
**Cons:** requires curation; can fail on queries unlike any exemplar.

> **Confidence trap:** the embedding router *always* returns some backend,
> even for "tell me a joke" (low similarity to everything).
> Always set a minimum similarity threshold and fall back when no backend scores high enough.


In [ ]:
from sentence_transformers import SentenceTransformer, util

EMBED_MODEL = SentenceTransformer('all-MiniLM-L6-v2', device='cpu')
print('Embedding model loaded.')

EXEMPLARS = {
    'sql': [
        'How many orders this month?',
        'What is the average revenue per customer?',
        'List all users who signed up last week',
        'Total orders placed in Q3',
        'What is our churn rate for Q2?',
        'Give me the revenue breakdown by quarter',
        'How many active subscriptions do we have?',
    ],
    'vector_store': [
        'How does our refund policy work?',
        'What features does the Pro plan include?',
        'What is the enterprise SLA guarantee?',
        'How do I contact support?',
        'What is the free tier storage limit?',
        'Explain the shipping policy',
    ],
    'web_search': [
        'What did our CEO announce yesterday?',
        'Latest news about our competitor',
        'Recent press release from our company',
        'What happened in our industry this week?',
        'Did our competitor launch a new product?',
    ],
    'api': [
        'What is the current TSLA stock price?',
        'What is the weather in New York today?',
        'Live stock quote for GOOGL',
        'Current temperature in London',
        'Real-time price of our preferred supplier stock',
    ],
}

# Pre-compute exemplar embeddings once
print('Pre-computing exemplar embeddings...')
EXEMPLAR_EMBEDS = {}
for backend, examples in EXEMPLARS.items():
    EXEMPLAR_EMBEDS[backend] = EMBED_MODEL.encode(
        examples, convert_to_tensor=True, show_progress_bar=False)
    print(f'  {backend}: {len(examples)} exemplars embedded')

print('Ready.')


In [ ]:
CONFIDENCE_THRESHOLD = 0.35  # queries below this score go to fallback


def embedding_router(query: str, threshold: float = CONFIDENCE_THRESHOLD) -> tuple:
    """
    Returns (backend, best_score, latency_ms).
    Returns ('fallback', score, latency_ms) when no backend exceeds threshold.
    """
    t0    = time.perf_counter()
    q_emb = EMBED_MODEL.encode(query, convert_to_tensor=True, show_progress_bar=False)

    best_backend, best_score = None, -1.0
    all_scores = {}
    for backend, embs in EXEMPLAR_EMBEDS.items():
        score = float(util.cos_sim(q_emb, embs).max())
        all_scores[backend] = score
        if score > best_score:
            best_backend, best_score = backend, score

    latency_ms = int((time.perf_counter() - t0) * 1000)

    if best_score < threshold:
        return 'fallback', best_score, latency_ms, all_scores

    return best_backend, best_score, latency_ms, all_scores


print('=== Embedding Router test suite ===\n')
print(f'{"Query":<55} {"Expected":<14} {"Got":<14} {"Score":<8} {"OK?"}')
print('-' * 100)

emb_correct = 0
emb_results = []
for q, expected, difficulty in TEST_CASES:
    got, score, lat, _ = embedding_router(q)
    ok = got == expected
    if ok:
        emb_correct += 1
    emb_results.append({'query': q, 'expected': expected, 'got': got,
                        'correct': ok, 'difficulty': difficulty,
                        'score': score, 'latency_ms': lat})
    mark = 'OK' if ok else '!'
    print(f'{q[:53]:<55} {expected:<14} {got:<14} {score:<8.3f} {mark}')

print()
print(f'Embedding router accuracy: {emb_correct}/{len(TEST_CASES)} = {emb_correct/len(TEST_CASES):.0%}')
print(f'LLM router accuracy      : {llm_correct}/{len(TEST_CASES)} = {llm_correct/len(TEST_CASES):.0%}')
print(f'Keyword router accuracy  : {kw_correct}/{len(TEST_CASES)} = {kw_correct/len(TEST_CASES):.0%}')


In [ ]:
# ── Confidence trap demo ──────────────────────────────────────────────────────
# The router always picks SOME backend. Show what happens with off-domain queries.

OFF_DOMAIN = [
    'Tell me a joke',
    'What is the capital of France?',
    'Write me a poem about autumn',
    'Translate this to Spanish: Hello world',
]

print('=== Confidence trap: off-domain queries ===\n')
print(f'{"Query":<50} {"Routed to":<14} {"Best score":<12} {"Safe?"}')
print('-' * 85)

for q in OFF_DOMAIN:
    backend, score, _, all_scores = embedding_router(q, threshold=0.0)  # no threshold
    safe_backend, safe_score, _, _ = embedding_router(q, threshold=CONFIDENCE_THRESHOLD)
    safe = safe_backend == 'fallback'
    print(f'{q:<50} {backend:<14} {score:<12.3f} {"CAUGHT" if safe else "LEAKED"}')

print()
print(f'Threshold = {CONFIDENCE_THRESHOLD}: queries below this score → "fallback" instead of a random backend.')
print()
print('Without a threshold: "Tell me a joke" gets silently routed to vector_store.')
print('The LLM retrieves a refund policy and tries to turn it into a joke.')
print('With threshold: "fallback" is returned and you can show a helpful error message.')


---
## 6. Router Comparison — Accuracy, Speed, Cost

| Router | Accuracy (easy) | Accuracy (hard) | Latency | Cost/query |
|---|---|---|---|---|
| Keyword | High | Low | < 1ms | $0 |
| Embedding | High | Medium-High | ~50ms | ~$0.0001 |
| LLM | High | High | 200–800ms | ~$0.001 |

**Practical advice:** start with keyword for known patterns, fall back to embedding for the rest.
Reserve LLM routing for genuinely ambiguous queries or when one wrong route is very costly.


In [ ]:
# Measure actual routing accuracy on easy vs hard queries
routers = [
    ('Keyword', kw_results),
    ('Embedding', emb_results),
    ('LLM (mock)', llm_results),
]

def accuracy(results, difficulty=None):
    if difficulty:
        subset = [r for r in results if r['difficulty'] == difficulty]
    else:
        subset = results
    if not subset:
        return 0.0
    return sum(1 for r in subset if r['correct']) / len(subset)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Accuracy by difficulty
router_names  = [r[0] for r in routers]
easy_accs     = [accuracy(r[1], 'easy') for r in routers]
hard_accs     = [accuracy(r[1], 'hard') for r in routers]

x = np.arange(len(router_names))
w = 0.35
axes[0].bar(x - w/2, easy_accs, w, color='#1565C0', alpha=0.85, label='Easy queries')
axes[0].bar(x + w/2, hard_accs, w, color='#E53935', alpha=0.85, label='Hard queries')
axes[0].set_xticks(x)
axes[0].set_xticklabels(router_names)
axes[0].set_ylim(0, 1.2)
axes[0].set_ylabel('Accuracy')
axes[0].set_title('Routing Accuracy', fontweight='bold')
axes[0].legend()
for i, (e, h) in enumerate(zip(easy_accs, hard_accs)):
    axes[0].text(i - w/2, e + 0.02, f'{e:.0%}', ha='center', fontsize=10)
    axes[0].text(i + w/2, h + 0.02, f'{h:.0%}', ha='center', fontsize=10)

# Latency
latency_ms = [0.5, 50, 250]  # rough representative values
colors_lat = ['#4CAF50', '#F9A825', '#E53935']
bars = axes[1].bar(router_names, latency_ms, color=colors_lat, alpha=0.85, width=0.5)
axes[1].set_ylabel('Latency (ms, log scale)')
axes[1].set_yscale('log')
axes[1].set_title('Routing Latency', fontweight='bold')
for bar, val in zip(bars, latency_ms):
    axes[1].text(bar.get_x() + bar.get_width()/2, val * 1.3,
                 f'{val}ms', ha='center', fontsize=10, fontweight='bold')

# Cost per 1000 queries (USD estimate)
cost_per_1k = [0.0, 0.10, 1.00]  # keyword=free, embedding=cheap, LLM=costs money
axes[2].bar(router_names, cost_per_1k, color=colors_lat, alpha=0.85, width=0.5)
axes[2].set_ylabel('Estimated cost per 1,000 queries (USD)')
axes[2].set_title('Routing Cost', fontweight='bold')
for bar, val in zip(axes[2].patches, cost_per_1k):
    axes[2].text(bar.get_x() + bar.get_width()/2,
                 val + 0.02, f'${val:.2f}', ha='center', fontsize=10, fontweight='bold')

plt.suptitle('Router Trade-offs: Accuracy vs. Latency vs. Cost', fontweight='bold', fontsize=12)
plt.tight_layout()
plt.show()

print('Key insight: the embedding router is often the best practical choice.')
print('It matches LLM accuracy on easy queries at 5x less latency and 10x less cost.')
print('Reserve LLM routing for genuinely ambiguous queries.')


---
## 7. Fan-out Pattern — Query Everything in Parallel

A contrarian alternative to routing: **don't route at all**.
Query all backends simultaneously, then let a reranker pick the best chunks.

```python
# Fan-out: parallel retrieval from all backends
results = parallel(
    vector_store.search(query),
    sql_db.query(query),
    web_search.search(query),
)
merged = rerank(flatten(results), query)
return merged[:top_k]
```

**Use fan-out when:**
- You don't know which backend has the answer
- Backends are fast and cheap (no per-call rate limits)
- A wrong route is more expensive than calling extras

**Avoid fan-out when:**
- A backend is slow or costly (web search APIs charge per call)
- Latency budget is tight (response time = slowest backend)
- Backends return very different formats that are hard to merge

> **Latency surprise:** fan-out is often **faster** than LLM routing!
> Serial: 500ms (LLM router) + 300ms (one backend) = 800ms.
> Parallel: max(300, 300, 300) + 100ms rerank = 400ms.
> You pay in dollars (3x calls), not in seconds.


In [ ]:
def fanout_retrieve(query: str, backends: list = None, top_k: int = 3) -> list:
    """
    Query all specified backends in parallel; return top-k merged results.
    Each result: (backend, result_text, score).
    Scoring: keyword overlap used as a simple relevance proxy.
    """
    if backends is None:
        backends = list(BACKENDS.keys())

    def fetch(backend_name):
        fn     = BACKENDS[backend_name]
        result, lat = fn(query)
        return backend_name, result, lat

    all_results = []
    t0 = time.perf_counter()
    with ThreadPoolExecutor(max_workers=len(backends)) as executor:
        futures = {executor.submit(fetch, b): b for b in backends}
        for future in as_completed(futures):
            backend_name, result, lat = future.result()
            # Simple relevance score: keyword overlap
            words = set(query.lower().split())
            score = len(words & set(result.lower().split())) / max(len(words), 1)
            all_results.append({'backend': backend_name, 'result': result,
                                 'score': score, 'backend_latency_ms': lat})

    total_ms = int((time.perf_counter() - t0) * 1000)
    all_results.sort(key=lambda x: x['score'], reverse=True)
    return all_results[:top_k], total_ms


fanout_queries = [
    'What was the total revenue in Q3?',
    'What did our CEO announce yesterday?',
    'How does our refund policy work?',
    'What is the current TSLA stock price?',
]

print('=== Fan-out pattern demo ===\n')
for q in fanout_queries:
    results, total_ms = fanout_retrieve(q, top_k=2)
    print(f'Query: "{q}"')
    print(f'  Total wall time: {total_ms}ms (all backends in parallel)')
    for r in results:
        print(f'  [{r["backend"]}] score={r["score"]:.2f}  {r["result"][:80]}...')
    print()

print('Fan-out returns the best result regardless of which backend had it.')
print('Downside: you called all backends even for trivial questions.')


---
## 8. Latency Math — Serial Router vs. Parallel Fan-out


In [ ]:
# ── Time each backend individually ──────────────────────────────────────────
test_query = 'What was the total revenue in Q3?'

backend_times = {}
for name, fn in BACKENDS.items():
    _, ms = fn(test_query)
    backend_times[name] = ms

# ── Time the fan-out ─────────────────────────────────────────────────────────
_, fanout_ms = fanout_retrieve(test_query)

# ── Simulated timings for serial routing approaches ───────────────────────────
# Serial = router time + winning backend time
correct_backend_time = backend_times['sql']  # correct route for this query

APPROACHES = [
    ('Keyword\nrouter', 0.5 + correct_backend_time, '< 1ms routing'),
    ('Embedding\nrouter', 50 + correct_backend_time, '~50ms routing'),
    ('LLM\nrouter', 400 + correct_backend_time, '~400ms routing'),
    ('Fan-out\n(parallel)', fanout_ms, f'{len(BACKENDS)} backends parallel'),
]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

# Left: total latency
labels  = [a[0] for a in APPROACHES]
times   = [a[1] for a in APPROACHES]
notes   = [a[2] for a in APPROACHES]
colors  = ['#4CAF50', '#F9A825', '#E53935', '#7B1FA2']

bars = ax1.bar(labels, times, color=colors, alpha=0.85, width=0.6)
ax1.set_ylabel('Total latency (ms)')
ax1.set_title('End-to-End Latency per Approach', fontweight='bold')
for bar, val, note in zip(bars, times, notes):
    ax1.text(bar.get_x() + bar.get_width()/2, val + 5,
             f'{val:.0f}ms', ha='center', fontsize=10, fontweight='bold')
    ax1.text(bar.get_x() + bar.get_width()/2, val/2,
             note, ha='center', fontsize=8, color='white')

# Right: breakdown stacked bar (routing overhead vs retrieval time)
routing_times   = [0.5, 50, 400, 0]
retrieval_times = [correct_backend_time, correct_backend_time,
                   correct_backend_time, fanout_ms]

ax2.bar(labels, routing_times,   color='#E53935', alpha=0.85, label='Routing overhead')
ax2.bar(labels, retrieval_times, bottom=routing_times,
        color='#1565C0', alpha=0.85, label='Retrieval time')
ax2.set_ylabel('Time (ms)')
ax2.set_title('Routing Overhead vs. Retrieval Time', fontweight='bold')
ax2.legend(loc='upper left')

plt.suptitle('Latency Math: Why Fan-out Can Beat LLM Routing',
             fontweight='bold', fontsize=12)
plt.tight_layout()
plt.show()

print(f'Individual backend latencies:')
for name, ms in backend_times.items():
    print(f'  {name:<14}: {ms}ms')
print(f'  Fan-out (parallel): {fanout_ms}ms  (bounded by slowest backend)')
print()
print('Key: LLM routing adds 400ms BEFORE retrieval.')
print('Fan-out adds 0ms routing cost but pays in dollars (N retrieval calls).')
print('For latency-sensitive apps: keyword router + fan-out > LLM router + single backend.')


---
## 9. Hybrid Router — Best of Both Worlds

Practical pattern used in production:

```
Query
  ↓
[Fast keyword/embedding router]
  ↓ HIGH CONFIDENCE     ↓ LOW CONFIDENCE
Single backend        Fan-out (all backends)
                          ↓
                       Rerank + pick top-k
```

- **Clear SQL query** → keyword router catches it instantly → single SQL call
- **Ambiguous query** → embedding router isn't sure → fan-out → reranker decides

This keeps costs low for the majority of queries while gracefully handling ambiguity.


In [ ]:
def hybrid_router(query: str,
                  embedding_threshold: float = 0.40,
                  top_k: int = 3) -> dict:
    """
    Step 1: Try keyword router (free).
    Step 2: Try embedding router (cheap).
    Step 3: If still uncertain, fan-out to all backends.
    """
    trace = []

    # ── Step 1: keyword router ────────────────────────────────────────────────
    kw_backend = keyword_router(query)
    # Only trust keyword router for its strongest signals
    HIGH_CONFIDENCE_KW = ['sql', 'api', 'web_search']  # not the default fallback
    if kw_backend in HIGH_CONFIDENCE_KW:
        result, lat = BACKENDS[kw_backend](query)
        trace.append(f'keyword_router → {kw_backend} (high confidence)')
        return {'path': 'keyword', 'backend': kw_backend,
                'result': result, 'trace': trace}

    # ── Step 2: embedding router ──────────────────────────────────────────────
    emb_backend, score, _, all_scores = embedding_router(query, threshold=embedding_threshold)
    trace.append(f'keyword_router → vector_store (default), '
                 f'embedding_router → {emb_backend} (score={score:.2f})')

    if emb_backend != 'fallback':
        result, lat = BACKENDS[emb_backend](query)
        trace.append(f'routed to {emb_backend}')
        return {'path': 'embedding', 'backend': emb_backend,
                'result': result, 'score': score, 'trace': trace}

    # ── Step 3: fan-out ───────────────────────────────────────────────────────
    trace.append(f'low confidence ({score:.2f} < {embedding_threshold}), using fan-out')
    results, total_ms = fanout_retrieve(query, top_k=top_k)
    trace.append(f'fan-out: {len(BACKENDS)} backends, {total_ms}ms')
    best = results[0] if results else None
    return {'path': 'fanout', 'backend': best['backend'] if best else 'none',
            'result': best['result'] if best else '', 'trace': trace}


print('=== Hybrid router demo ===\n')
hybrid_demos = [
    ('How many orders did we get this month?',         'sql'),
    ('What is the TSLA stock price?',                  'api'),
    ('What did our CEO announce yesterday?',            'web_search'),
    ('How does our refund policy work?',                'vector_store'),
    ('Show me our numbers for last quarter',            'sql'),
    ('What is going on with enterprise adoption?',      'ambiguous'),
]
for q, expected in hybrid_demos:
    result = hybrid_router(q)
    path   = result['path']
    backend = result['backend']
    print(f'Query  : "{q}"')
    print(f'  Expected  : {expected}')
    print(f'  Path taken: {path} → {backend}')
    for step in result['trace']:
        print(f'    {step}')
    print()

print('Observation: clear queries go through keyword/embedding in < 60ms.')
print('Ambiguous queries trigger fan-out and let the reranker decide.')


---
## 10. Tool-call Routing — Letting the LLM Choose

Instead of a separate routing step, describe each backend as a **tool** and let the LLM
pick which tool to call. This is how most modern agentic RAG systems actually work.

**Difference from the LLM router:**
- LLM router: picks a backend name → you write the query
- Tool calling: picks a backend **and** generates the query parameters in one step

```python
# LLM router: two steps
backend = llm_router(user_query)          # step 1: pick backend
result  = BACKENDS[backend](user_query)   # step 2: you write the query

# Tool calling: one step
tool_call = llm.chat(user_query, tools=tools)  # LLM picks AND queries
result    = execute(tool_call)                  # you just run it
```

> **Tool descriptions are your router prompt.**
> Treat each description like a one-line job posting.
> Overlapping descriptions = unpredictable routing.
> Be specific about which queries each tool should attract — and which it should avoid.


In [ ]:
TOOLS = [
    {
        'name': 'search_documentation',
        'description': (
            'Search internal product documentation, policies, and how-to guides. '
            'Use for questions about features, refund policy, SLA details, '
            'shipping, or pricing tiers. '
            'Do NOT use for numeric business metrics or current events.'
        ),
        'input_schema': {
            'type': 'object',
            'properties': {
                'query': {'type': 'string',
                          'description': 'Natural language search query'}
            },
            'required': ['query'],
        },
    },
    {
        'name': 'query_sql_database',
        'description': (
            'Query the company SQL database for structured business data. '
            'Use for questions about order counts, revenue totals, customer data, '
            'churn rates, and any numeric business metric. '
            'Do NOT use for qualitative or policy questions.'
        ),
        'input_schema': {
            'type': 'object',
            'properties': {
                'query': {'type': 'string',
                          'description': 'Natural language description of the data needed'}
            },
            'required': ['query'],
        },
    },
    {
        'name': 'search_web',
        'description': (
            'Search the web for recent news, press releases, and current events. '
            'Use for questions about announcements, competitor activity, or anything '
            'that happened in the last few days. '
            'Do NOT use for internal company data or static documentation.'
        ),
        'input_schema': {
            'type': 'object',
            'properties': {
                'query': {'type': 'string',
                          'description': 'Web search query string'}
            },
            'required': ['query'],
        },
    },
    {
        'name': 'call_live_api',
        'description': (
            'Fetch real-time live data from external APIs: stock prices, weather, '
            'exchange rates. Use ONLY for questions explicitly asking for current, '
            'live, or real-time values. Do NOT use for historical data.'
        ),
        'input_schema': {
            'type': 'object',
            'properties': {
                'query': {'type': 'string',
                          'description': 'What real-time data to fetch (e.g., "TSLA stock price")'}
            },
            'required': ['query'],
        },
    },
]

TOOL_TO_BACKEND = {
    'search_documentation': 'vector_store',
    'query_sql_database':   'sql',
    'search_web':           'web_search',
    'call_live_api':        'api',
}


def tool_call_router(user_query: str) -> dict:
    """Use Claude tool use to pick and parameterise the backend in one step."""
    if USE_REAL_LLM and ANTHROPIC_AVAILABLE:
        client = _anthropic_module.Anthropic()
        resp   = client.messages.create(
            model='claude-haiku-4-5-20251001',
            max_tokens=200,
            tools=TOOLS,
            messages=[{'role': 'user', 'content': user_query}],
        )

        tool_use = next((b for b in resp.content if b.type == 'tool_use'), None)
        if not tool_use:
            return {'tool': None, 'backend': 'none', 'query': user_query,
                    'result': 'No tool selected by LLM.'}

        tool_name   = tool_use.name
        tool_query  = tool_use.input.get('query', user_query)
        backend     = TOOL_TO_BACKEND.get(tool_name, 'vector_store')
        result, _   = BACKENDS[backend](tool_query)
        return {'tool': tool_name, 'backend': backend,
                'query': tool_query, 'result': result}

    # Mock: use the LLM router mock (same routing decisions)
    backend, _ = llm_router(user_query)
    tool_name  = next(t for t, b in TOOL_TO_BACKEND.items() if b == backend)
    # Mock query formulation: strip filler words
    mock_query = re.sub(r'(what is|how does|tell me|show me|what was)\s+', '',
                        user_query.lower()).strip()
    result, _  = BACKENDS[backend](mock_query)
    return {'tool': tool_name, 'backend': backend,
            'query': mock_query, 'result': result}


print('=== Tool-call routing demo ===\n')
tc_demos = [
    'How many orders did we get this month?',
    'What features does the Pro plan include?',
    'What did our CEO announce yesterday?',
    'What is the current TSLA stock price?',
    'Show me our numbers for last quarter',
]
for q in tc_demos:
    r = tool_call_router(q)
    print(f'Query    : "{q}"')
    print(f'Tool     : {r["tool"]}')
    print(f'Backend  : {r["backend"]}')
    print(f'Sub-query: "{r["query"]}"')
    print(f'Result   : {r["result"][:100]}...')
    print()

print('Key difference from LLM router:')
print('  LLM router returns: "sql"')
print('  Tool calling returns: {tool: "query_sql_database", query: "monthly orders count"}')
print('  The LLM formulates the backend-specific query in one step.')


---
## 11. Decision Framework — Which Router Should You Pick?

Ask in this order:

```
1. Can I predict all the query patterns in advance?
   YES → keyword router (free, deterministic)
   NO  ↓

2. Can I write 5-10 exemplars per backend?
   YES → embedding router (~50ms, cheap)
   NO  ↓

3. Do I need the LLM to also formulate the backend query?
   YES → tool calling (most powerful, most expensive)
   NO  → LLM router prompt (picks backend only)

Also ask: what is the cost of routing wrong?
  LOW cost  → lean cheap (keyword/embedding)
  HIGH cost → lean accurate (LLM/tool calling)
```

**Real production pattern:**
- **Obvious queries** (strong keyword signals) → keyword router → single backend
- **Most queries** → embedding router → single backend
- **Ambiguous/high-stakes queries** → LLM/tool calling or fan-out
- **Unknown query types** → fan-out + rerank

> **The cost of wrong routing should set your routing budget.**
> If wrong means a hallucinated revenue number shown to the CFO: pay for LLM routing.
> If wrong means a slightly worse FAQ answer: keyword routing is fine.


In [ ]:
def recommend_router(query_patterns_known: bool,
                     exemplars_available: bool,
                     need_backend_query: bool,
                     wrong_route_cost: str) -> dict:
    """
    Decision framework from the blog.
    wrong_route_cost: 'low', 'medium', 'high'
    """
    if query_patterns_known:
        router = 'keyword'
        reason = 'All patterns predictable; keyword matching is free and deterministic.'
    elif exemplars_available:
        router = 'embedding'
        reason = 'Can curate exemplars; ~50ms, cheap, handles natural language variants.'
    elif need_backend_query:
        router = 'tool_calling'
        reason = 'Need LLM to pick backend AND formulate the query in one step.'
    else:
        router = 'llm_router'
        reason = 'Diverse queries; LLM understands intent; just need backend selection.'

    if wrong_route_cost == 'high' and router in ('keyword', 'embedding'):
        upgrade = 'llm_router or tool_calling'
        warning = f'Cost of wrong route is HIGH — upgrade from {router} to {upgrade}'
    else:
        warning = None

    return {'router': router, 'reason': reason, 'warning': warning}


print('=== Decision framework examples ===\n')
scenarios = [
    {
        'name': 'Finance dashboard (narrow query vocabulary)',
        'query_patterns_known': True, 'exemplars_available': True,
        'need_backend_query': False, 'wrong_route_cost': 'medium',
    },
    {
        'name': 'General customer support bot',
        'query_patterns_known': False, 'exemplars_available': True,
        'need_backend_query': False, 'wrong_route_cost': 'low',
    },
    {
        'name': 'Executive dashboard (shows CFO a revenue number)',
        'query_patterns_known': False, 'exemplars_available': True,
        'need_backend_query': False, 'wrong_route_cost': 'high',
    },
    {
        'name': 'Agentic research assistant (needs SQL, web, docs)',
        'query_patterns_known': False, 'exemplars_available': True,
        'need_backend_query': True, 'wrong_route_cost': 'medium',
    },
]
for s in scenarios:
    rec = recommend_router(
        s['query_patterns_known'], s['exemplars_available'],
        s['need_backend_query'], s['wrong_route_cost'])
    print(f'Scenario: {s["name"]}')
    print(f'  Recommended: {rec["router"]}')
    print(f'  Reason     : {rec["reason"]}')
    if rec['warning']:
        print(f'  WARNING    : {rec["warning"]}')
    print()


---
## Key Takeaways

1. **Wrong routing causes hallucination.** A SQL question sent to the vector store
   retrieves a vague marketing sentence; the LLM fills in a plausible-sounding number.
   No error is raised. The user gets a confident wrong answer. Routing is the gate.

2. **Three routers, three trade-offs.** Keyword (free, brittle) → embedding (fast, good) →
   LLM (accurate, expensive). Don't jump to LLM routing until keyword and embedding fail.

3. **The confidence trap is real.** An embedding router always returns some backend,
   even for off-domain queries. Set a minimum similarity threshold and fall back gracefully
   when no backend scores high enough.

4. **Fan-out is surprisingly fast.** Parallel retrieval across N backends is bounded by
   the slowest backend — often faster than serial LLM routing + one retrieval.
   The cost is in dollars, not latency.

5. **Tool calling combines routing and query formulation.** The LLM doesn't just pick a
   backend — it also writes the backend-specific query. Tool descriptions are your router
   prompt; overlapping descriptions cause unpredictable routing.

6. **The cost of routing wrong sets your routing budget.** Hallucinated FAQ answer:
   use keyword routing, it's free. Hallucinated revenue number shown to the CFO:
   pay for LLM routing and log every decision.

7. **Log everything.** Query, chosen backend, and model reasoning. Without logs,
   debugging an LLM router when a user complains means reading tea leaves.

---

*Up next — Module 11: Evaluation. How do you actually know any of this is working?*
